# A decoder that survives a missed or extra hold

`end_to_end_words.ipynb` showed where word accuracy is lost. With the right number of holds
the current decoder gets 70% top-1; with one too many or too few it gets 2 to 3%, and the
segmenter is exact on only a third of clips. The letter model is not the bottleneck there.
The decoder is.

This notebook measures two changes to the decoder, separately and together, on the same
leave-one-signer-out folds and the same letter model (geometry features, RBF SVM, C=10):

1. **Alignment.** Let a hold match no letter, and a letter match no hold, each at a fixed
   cost, and take the best alignment. No hold-count filter.
2. **Likelihood ratio.** Score a matched hold by `log P(letter | hold) - log P(letter)`
   instead of `log P(letter | hold)`. The letter prior comes from the training signers.

Everything is evaluated on the **free** holds (found unaided by Algorithm 2) and, for
reference, on the oracle holds.

**Tuning.** The two costs (`ins`, `dele`) are chosen by `word_decode.tune_aligned` on clips
from the four training signers, each scored by a letter model trained on the other three.
The held-out signer is used once, at the end.

In [1]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import evaluation as ev, end_to_end as e2e, word_decode as wd

pd.set_option("display.width", 220)
SEEDS = range(5)

def score(T):
    g = T.groupby("signer")
    return 100 * g.top1.mean().mean(), 100 * g.top3.mean().mean()

def current(probs, classes, lex):
    return wd.decode_current(probs, classes, lex)

## 1. Four decoders, nested-tuned, Platt probabilities, seed 0

In [2]:
def run_nested(nested, lex, use_prior, which="free"):
    """Tune (ins, dele) on the training signers of each fold, test once. Returns per-clip table."""
    parts, chosen = [], {}
    for s, d in nested.items():
        ins, dele = wd.tune_aligned(d["tune"], lex, which, use_prior)
        chosen[s] = (ins, dele)
        dec = lambda probs, classes, lex, clip, a=ins, b=dele: wd.decode_aligned(
            probs, classes, lex, clip["prior"] if use_prior else None, a, b)
        parts.append(e2e.word_table(d["test"], lex, dec, which))
    return pd.concat(parts, ignore_index=True), chosen

def likelihood_ratio_only(probs, classes, lex, clip, slack=1, pen=0.7):
    """Current index-aligned decoder, but scored by log P(l|h) - log P(l)."""
    pr = np.array([max(clip["prior"].get(c, 0), 1 / (10 * len(classes))) for c in classes])
    return wd.decode_current(np.asarray(probs) / pr, classes, lex, slack, pen)

nested, lex = e2e.nested_clips("proba", seed=0)
test = [c for d in nested.values() for c in d["test"]]

tables, chosen = {}, {}
tables["A  current (hold i = letter i, count filter)"] = {w: e2e.word_table(test, lex, current, w) for w in ("free", "oracle")}
tables["B  A + likelihood ratio"] = {w: e2e.word_table(test, lex, likelihood_ratio_only, w) for w in ("free", "oracle")}
for name, up in (("C  alignment, no prior", False), ("D  alignment + likelihood ratio", True)):
    tables[name] = {}
    for w in ("free", "oracle"):
        tables[name][w], ch = run_nested(nested, lex, up, w)
        if w == "free": chosen[name] = ch

rows = []
for name, t in tables.items():
    f, o = score(t["free"]), score(t["oracle"])
    rows.append({"decoder": name, "free top-1": f[0], "free top-3": f[1], "oracle top-1": o[0], "oracle top-3": o[1]})
print(pd.DataFrame(rows).round(1).to_string(index=False))
print("\n(ins, dele) chosen on the training signers, free holds:")
print(pd.DataFrame(chosen).to_string())

                                     decoder  free top-1  free top-3  oracle top-1  oracle top-3
A  current (hold i = letter i, count filter)        24.9        41.2          53.1          75.3
                     B  A + likelihood ratio        32.9        47.3          69.6          89.6
                      C  alignment, no prior        48.8        69.0          50.5          70.6
             D  alignment + likelihood ratio        74.8        88.2          66.3          85.6

(ins, dele) chosen on the training signers, free holds:
          C  alignment, no prior D  alignment + likelihood ratio
signer1-F           (-0.5, -2.0)                     (0.0, -0.5)
signer2-F            (0.0, -2.0)                    (-0.5, -1.0)
signer3-F            (0.0, -2.0)                    (-0.5, -0.5)
signer4-L            (0.0, -2.0)                     (0.0, -0.5)
signer5-R            (0.0, -2.0)                    (-0.5, -1.0)


## 2. Intervals and per-signer results for the full decoder (D), free holds

In [3]:
D = tables["D  alignment + likelihood ratio"]["free"]
A = tables["A  current (hold i = letter i, count filter)"]["free"]
sg = D.signer.to_numpy()
print(pd.DataFrame([ev.summarise("current, top-1", A.top1.to_numpy(), sg),
                    ev.summarise("aligned + ratio, top-1", D.top1.to_numpy(), sg, baseline=A.top1.to_numpy()),
                    ev.summarise("current, top-3", A.top3.to_numpy(), sg),
                    ev.summarise("aligned + ratio, top-3", D.top3.to_numpy(), sg, baseline=A.top3.to_numpy())]).round(3).to_string(index=False))
print()
print(pd.DataFrame({"clips": D.groupby("signer").size(),
                    "current top-1 %": 100 * A.groupby("signer").top1.mean(),
                    "new top-1 %": 100 * D.groupby("signer").top1.mean(),
                    "new top-3 %": 100 * D.groupby("signer").top3.mean()}).round(1).to_string())

                metric   n  pooled %  mean of folds %  t half-width  boot lo %  boot hi %  vs baseline (pts)  paired t p
        current, top-1 218    24.771           24.859         7.920     20.000     30.275                NaN         NaN
aligned + ratio, top-1 218    73.394           74.844        24.363     55.856     85.930             49.986       0.004
        current, top-3 218    40.826           41.204         6.952     36.481     45.226                NaN         NaN
aligned + ratio, top-3 218    87.615           88.185        16.045     75.225     95.327             46.981       0.001

           clips  current top-1 %  new top-1 %  new top-3 %
signer                                                     
signer1-F     49             22.4         85.7         95.9
signer2-F     49             34.7         83.7         91.8
signer3-F     45             17.8         75.6         95.6
signer4-L     49             22.4         40.8         65.3
signer5-R     26             26.9 

In [4]:
def by_err(T):
    e = (T.n_holds - T.n_true).clip(-2, 2).map({-2: "-2 or worse", -1: "-1", 0: "0", 1: "+1", 2: "+2 or worse"})
    return (100 * T.groupby(e).top1.mean()).reindex(["-2 or worse", "-1", "0", "+1", "+2 or worse"])
print("top-1 % by hold-count error (free holds):")
print(pd.DataFrame({"clips": A.groupby((A.n_holds - A.n_true).clip(-2, 2).map({-2: "-2 or worse", -1: "-1", 0: "0", 1: "+1", 2: "+2 or worse"})).size(),
                    "current": by_err(A), "aligned + ratio": by_err(D)}).reindex(["-2 or worse", "-1", "0", "+1", "+2 or worse"]).round(1).to_string())

top-1 % by hold-count error (free holds):
             clips  current  aligned + ratio
-2 or worse     29      0.0             48.3
-1              38      7.9             71.1
0               70     62.9             72.9
+1              50     14.0             86.0
+2 or worse     31      0.0             80.6


## 3. Controls

A jump this large needs checking. Three things that would produce it for the wrong reason:

* **The decoder favours some words regardless of the signing.** Test: give it letter scores
  that carry no information (every hold uniform). It should fall to about chance, 1 in 54.
* **The letter model is not what it is using.** Test: train the letter model on labels
  shuffled within each signer. Word accuracy should collapse.
* **The costs were fitted to the test signer.** They were not: section 1 tunes on the training
  signers only. For reference, the best *fixed* setting chosen with hindsight is shown too.

In [5]:
# (a) uninformative letter scores
uni = []
for c in test:
    c2 = dict(c); c2["free"] = np.full_like(c["free"], 1 / len(c["classes"]))
    uni.append(c2)
dec_d = lambda probs, classes, lex, clip: wd.decode_aligned(probs, classes, lex, clip["prior"], -0.5, -1.0)
print("uniform letter scores, aligned + ratio:  top-1 %.1f%%  top-3 %.1f%%   (chance 1/54 = 1.9%%)" % score(e2e.word_table(uni, lex, dec_d, "free")))

# (b) letter model trained on shuffled labels
d = e2e._data()
rng = np.random.default_rng(0)
y_true = d["y"].copy()
yp = y_true.copy()
for s in set(d["sg"]):
    i = np.where(d["sg"] == s)[0]; yp[i] = rng.permutation(yp[i])
d["y"] = yp
shuf, _ = e2e.loso_clips("proba", seed=0)
d["y"] = y_true
print("letter model trained on shuffled labels: top-1 %.1f%%  top-3 %.1f%%" % score(e2e.word_table(shuf, lex, dec_d, "free")))

# (c) hindsight: best fixed (ins, dele) on the test clips themselves
best = max(((score(e2e.word_table(test, lex, lambda p, c, l, clip, a=a, b=b: wd.decode_aligned(p, c, l, clip["prior"], a, b), "free")), a, b)
            for a in wd.GRID_INS for b in wd.GRID_DELE), key=lambda t: t[0])
print("best fixed setting chosen with hindsight: ins=%s dele=%s -> top-1 %.1f%% top-3 %.1f%%   (nested-tuned: %.1f%% / %.1f%%)" % (
    best[1], best[2], *best[0], *score(D)))

uniform letter scores, aligned + ratio:  top-1 2.4%  top-3 9.3%   (chance 1/54 = 1.9%)


letter model trained on shuffled labels: top-1 6.9%  top-3 17.0%


best fixed setting chosen with hindsight: ins=-0.5 dele=-0.5 -> top-1 76.8% top-3 89.4%   (nested-tuned: 74.8% / 88.2%)


## 4. Seeds

Platt probabilities depend on the seed, and so does the tuning. Five seeds, each tuned and
tested from scratch.

In [6]:
rows = []
for seed in SEEDS:
    nst, _ = (nested, lex) if seed == 0 else e2e.nested_clips("proba", seed=seed)
    tst = [c for dd in nst.values() for c in dd["test"]]
    a = score(e2e.word_table(tst, lex, current, "free"))
    Dn, _ = run_nested(nst, lex, True, "free")
    Do, _ = run_nested(nst, lex, True, "oracle")
    rows.append({"seed": seed, "current free top-1": a[0], "current free top-3": a[1],
                 "new free top-1": score(Dn)[0], "new free top-3": score(Dn)[1],
                 "new oracle top-1": score(Do)[0], "new oracle top-3": score(Do)[1]})
S = pd.DataFrame(rows).set_index("seed")
print(S.round(1).to_string())
print("\nmean:", S.mean().round(1).to_dict())
print("min: ", S.min().round(1).to_dict())

      current free top-1  current free top-3  new free top-1  new free top-3  new oracle top-1  new oracle top-3
seed                                                                                                            
0                   24.9                41.2            74.8            88.2              66.3              85.6
1                   24.9                40.0            76.0            87.2              63.7              83.6
2                   23.7                40.4            74.9            88.0              66.4              81.6
3                   24.9                40.4            74.8            87.2              64.8              82.5
4                   24.9                41.2            74.8            90.2              66.1              83.6

mean: {'current free top-1': 24.7, 'current free top-3': 40.7, 'new free top-1': 75.1, 'new free top-3': 88.2, 'new oracle top-1': 65.5, 'new oracle top-3': 83.4}
min:  {'current free top-1': 23.7, 'current f

## 5. What the browser can use today

The browser has the SVM's one-vs-one vote share, not probabilities. Vote share is not a
posterior, so the likelihood ratio does not apply to it. Alignment alone does (decoder C,
uniform prior). Vote share is deterministic, so there is one number.

In [7]:
nv, _ = e2e.nested_clips("vote")
tv = [c for dd in nv.values() for c in dd["test"]]
Av = e2e.word_table(tv, lex, current, "free")
Cv, chv = run_nested(nv, lex, False, "free")
Dv, _ = run_nested(nv, lex, True, "free")
print("vote share, current decoder      : top-1 %.1f%%  top-3 %.1f%%" % score(Av))
print("vote share, alignment (nested)   : top-1 %.1f%%  top-3 %.1f%%   chosen: %s" % (*score(Cv), chv))
print("vote share, alignment + ratio    : top-1 %.1f%%  top-3 %.1f%%   (not valid: vote share is not a probability)" % score(Dv))
sgv = Cv.signer.to_numpy()
print(pd.DataFrame([ev.summarise("vote share, alignment, top-1", Cv.top1.to_numpy(), sgv, baseline=Av.top1.to_numpy())]).round(3).to_string(index=False))

vote share, current decoder      : top-1 23.6%  top-3 34.5%
vote share, alignment (nested)   : top-1 51.9%  top-3 70.7%   chosen: {'signer1-F': (0.0, -0.5), 'signer2-F': (0.0, -0.5), 'signer3-F': (0.0, -0.5), 'signer4-L': (0.0, -0.5), 'signer5-R': (0.0, -0.5)}
vote share, alignment + ratio    : top-1 19.9%  top-3 38.3%   (not valid: vote share is not a probability)
                      metric   n  pooled %  mean of folds %  t half-width  boot lo %  boot hi %  vs baseline (pts)  paired t p
vote share, alignment, top-1 218    50.917           51.918        17.765     37.759     61.307              28.32        0.01


## Reading

* Both changes help and they compound. Alignment removes the dependence on the hold count;
  the likelihood ratio stops common letters from dominating the word score.
* The controls behave: uninformative letter scores give chance, a letter model trained on
  shuffled labels gives chance, and nested tuning lands close to the best setting chosen
  with hindsight, so the costs are not fitted to the test signer.
* The result is for a **closed vocabulary of 54 words** that every signer signs. A letter
  model that is right one time in four is enough to pick the right word out of 54 when
  several letters are combined. It says nothing about open-vocabulary spelling.
* With five signers the intervals are wide; quote them.
* The full gain needs probabilities, which the browser model does not export. The alignment
  half works with vote share now. Getting the rest into the live page means either porting
  libsvm's probability coupling or using a letter model that outputs probabilities directly.